In [ ]:
import pandas as pd

df = pd.read_csv("Obesity.csv")
print(df.shape)
print(df.info())

In [ ]:
print(df['Gender'].value_counts())

In [ ]:
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

In [ ]:
print(df['FAVC'].value_counts())

In [ ]:
df['FAVC'] = df['FAVC'].map({"yes": 1, "no": 0})

In [ ]:
print(df['SCC'].unique())


In [ ]:
df['SCC'] = df['SCC'].map({"no":0, "yes": 1})

In [ ]:
print(df['SMOKE'].unique())

In [ ]:
df['SMOKE'] = df['SMOKE'].map({"no":0, "yes": 1})

In [ ]:
print(df['CH2O'].info())

In [ ]:
print(df['family_history_with_overweight'].value_counts())

In [ ]:
df['family_history_with_overweight'] = df['family_history_with_overweight'].map({"no": 0, "yes": 1})

In [ ]:
print(df['CAEC'].value_counts())

In [ ]:
print(df['MTRANS'].value_counts())

In [ ]:
print(df['NObeyesdad'].value_counts())

In [ ]:
print(df.info())

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns

plt.figure(figsize=(14,10))
sns.boxplot(x = 'NObeyesdad', y = 'Weight', data = df)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2 ,random_state=42)

In [ ]:
print(X_train.select_dtypes(include='float64').columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])


In [ ]:
print(X_train_scaled.select_dtypes(include='str').columns)

In [ ]:
cols = ['CALC', 'CAEC', 'MTRANS']

X_train_scaled = pd.get_dummies(X_train_scaled, columns= cols, drop_first=True)
X_test_scaled = pd.get_dummies(X_test_scaled, columns= cols, drop_first=True)

X_train_scaled, X_test_scaled = X_train_scaled.align(X_test_scaled, join = 'left', axis = 1, fill_value=0)

In [ ]:
print(X_train_scaled.head())


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42), 
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    test_acc =  accuracy_score(y_test, model.predict(X_test_scaled))
    train_acc =  accuracy_score(y_train, model.predict(X_train_scaled))
    print(f"{name}: Test: {test_acc}, Train: {train_acc}")


In [ ]:
from sklearn.metrics import classification_report
for name, model in models.items():
    print(f"{name}: {classification_report(y_test, model.predict(X_test_scaled))}")

In [ ]:
from sklearn.model_selection import cross_val_score

score = cross_val_score(models['Random Forest'], X_train_scaled, y_train, cv = 5, scoring = 'accuracy')
print(f"mean: {score.mean()} +/- {score.std()}")

In [ ]:
rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2,5]
}

In [ ]:
from sklearn.model_selection import GridSearchCV

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42), 
    rf_param_grid,
    cv = 5, 
    n_jobs= -1,
    scoring = 'accuracy'
)

rf_grid.fit(X_train_scaled, y_train)
print(rf_grid.best_params_)
print(rf_grid.best_score_)
best_model = rf_grid.best_estimator_
print(best_model.score(X_test_scaled, y_test))

In [ ]:
print(classification_report(y_test, best_model.predict(X_test_scaled)))

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = best_model.predict(X_test_scaled)
labels = ['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot = True, fmt = 'd', xticklabels= labels , yticklabels= labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
import joblib

joblib.dump(best_model, "Obestiy_model.pkl")

In [ ]:
loaded_model = joblib.load('Obestiy_model.pkl')
print(loaded_model.score(X_test_scaled, y_test))